# FINS exp53 + steppers — Kuramoto-Sivashinsky weak runs and BO / KS operator training (Colab, GPU)

Two jobs, both on the GPU:
* cell 2: the 50 weak-form runs of the registered Kuramoto-Sivashinsky study (`exp53_ks.py --parts weak`: 5 noise
  levels x seeds 42-51, exact solver, no training);
* cell 3: five FNO steppers for the learned arms, Benjamin-Ono seeds 43, 44 and Kuramoto-Sivashinsky seeds 42, 43, 44
  (exp39's recipe: 1024 training trajectories, 400 epochs, 64 modes), each followed by pointwise FINS through it on the
  development data (seed 1007) as a check.

Upload the newest `fins_repo_full.zip` from the chat to My Drive first (the setup cell takes the newest
`fins_repo_full*.zip` containing `experiments/exp53_ks.py`). Outputs and checkpoints go to `MyDrive/fins_runs/results`;
after a disconnect rerun cell 1 and the interrupted cell: finished runs are skipped and a training resumes from its
partial state. Cell 4 shows progress; cell 5 zips the outputs and the five checkpoints for download.

In [ ]:
# 1. setup: Drive, repository from the newest fins_repo_full*.zip on Drive that contains exp53, dependencies
import os, sys, subprocess, zipfile, glob, time, shutil
from google.colab import drive
drive.mount('/content/drive')
os.environ['FINS_RESULTS_DIR'] = '/content/drive/MyDrive/fins_runs/results'
os.makedirs(os.environ['FINS_RESULTS_DIR'], exist_ok=True)
cands = sorted(glob.glob('/content/drive/MyDrive/fins_repo_full*.zip') + glob.glob('/content/drive/MyDrive/*/fins_repo_full*.zip'),
               key=os.path.getmtime, reverse=True)
ZIP_PATH = None
for z in cands:
    has = 'fins_repo/experiments/exp53_ks.py' in zipfile.ZipFile(z).namelist()
    print(f'{z}  {time.ctime(os.path.getmtime(z))}  {os.path.getsize(z)} bytes  exp53: {has}')
    if has and ZIP_PATH is None:
        ZIP_PATH = z
if ZIP_PATH is None:
    raise SystemExit('no fins_repo_full*.zip on Drive contains experiments/exp53_ks.py: upload the newest zip from '
                     'the chat to My Drive and rerun this cell')
print('using', ZIP_PATH)
shutil.rmtree('/content/fins', ignore_errors=True)
with zipfile.ZipFile(ZIP_PATH) as z:
    z.extractall('/content/fins')
REPO = '/content/fins/fins_repo'
EXPDIR = os.path.join(REPO, 'experiments')
R = os.environ['FINS_RESULTS_DIR']
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'neuraloperator'], check=False)
import torch
print('torch', torch.__version__, '| cuda:', torch.cuda.is_available(),
      torch.cuda.get_device_name(0) if torch.cuda.is_available() else '')
if not torch.cuda.is_available():
    print('NO GPU: Runtime -> Change runtime type -> T4 GPU, then rerun this cell')

def stream(cmd):
    t0 = time.time()
    with subprocess.Popen(cmd, cwd=EXPDIR, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1) as p:
        for line in p.stdout:
            print(line, end='', flush=True)
    print(f'exit {p.returncode} after {time.time() - t0:.0f} s')

In [ ]:
# 2. the registered KS weak-form runs: 50 runs (5 noise levels x seeds 42-51); one line per finished run
stream([sys.executable, '-u', 'exp53_ks.py', 'run', '--parts', 'weak', '--runs', os.path.join(R, 'ks_runs')])

In [ ]:
# 3. five FNO steppers (BO 43, 44; KS 42, 43, 44) into FINS_RESULTS_DIR, each checked by pointwise FINS on data 1007
os.makedirs(os.path.join(R, 'steppers'), exist_ok=True)
for pde, s in [('bo', 43), ('bo', 44), ('ks', 42), ('ks', 43), ('ks', 44)]:
    out = os.path.join(R, 'steppers', f'pointwise_{pde}_stepper_s{s}.npz')
    if os.path.exists(out):
        print('done', out); continue
    log = out.replace('.npz', '.log')
    cmd = [sys.executable, '-u', 'exp39_localised.py', '--pde', pde, '--op', 'stepper', '--train-ic', 'mixed',
           '--train-if-missing', '--seed', str(s), '--data-seed', '1007', '--ic', 'periodic', '--n', '10', '--K', '16',
           '--every', '16', '--interior', '--ut', 'fd', '--save', out]
    t0 = time.time()
    with open(log, 'a') as fh:
        rc = subprocess.call(cmd, cwd=EXPDIR, stdout=fh, stderr=subprocess.STDOUT)
    lines = [l for l in open(log).read().splitlines() if l.strip()]
    print(pde, s, 'exit', rc, f'{time.time() - t0:.0f} s |', lines[-2][:120] if len(lines) > 1 else '')

In [ ]:
# 4. progress (safe while cell 2 or 3 runs, or after it stopped)
for d in sorted(glob.glob(os.path.join(R, 'ks_runs', '*'))):
    done = sorted(os.path.basename(f) for f in glob.glob(os.path.join(d, 'weak_ks_*.npz')))
    print(os.path.basename(d), len(done), 'of 10')
for f in sorted(glob.glob(os.path.join(R, 'steppers', '*.log'))):
    ep = [l for l in open(f).read().splitlines() if 'epoch' in l]
    print(os.path.basename(f), '|', ep[-1].strip()[:110] if ep else 'starting')
print(sorted(os.path.basename(p) for p in glob.glob(os.path.join(R, 'S_theta_*_mixed_stepper.pt'))
             if '_bo_' in p or '_ks_' in p))

In [ ]:
# 5. zip: KS runs (npz + logs), the stepper checks and the five checkpoints; download, a copy stays on Drive
from google.colab import files
stage = '/content/fins_exp53_steppers'
shutil.rmtree(stage, ignore_errors=True)
shutil.copytree(os.path.join(R, 'ks_runs'), os.path.join(stage, 'ks_runs'))
shutil.copytree(os.path.join(R, 'steppers'), os.path.join(stage, 'steppers'))
for pde, s in [('bo', 43), ('bo', 44), ('ks', 42), ('ks', 43), ('ks', 44)]:
    ck = os.path.join(R, f'S_theta_{pde}_s{s}_mixed_stepper.pt')
    if os.path.exists(ck):
        shutil.copy(ck, os.path.join(stage, 'steppers'))
arc = shutil.make_archive('/content/fins_exp53_steppers', 'zip', stage)
shutil.copy(arc, '/content/drive/MyDrive/fins_exp53_steppers.zip')
print(arc, os.path.getsize(arc), 'bytes; a copy is on Drive')
files.download(arc)